In [1]:
import numpy as np
from scipy.signal import find_peaks

TAU = (1 + 5**0.5) / 2
rng = np.random.default_rng(42)

# ---------------- parameters (tweak here) ----------------
E_LO, E_HI = 1.0, 8.0        # analysis window, meV: above noise, below the dense mush
RATIO_LO, RATIO_HI = 1.10, 3.0
TOL = 0.03                    # +-3% counts as "at" a target ratio
N_NULL = 20000                # bootstrap draws

targets = {"tau": TAU, "tau^2": TAU**2}
fib = {"3/2": 1.5, "5/3": 5/3, "8/5": 1.6, "2/1": 2.0}   # reported, not scored

def features(path, label):
    f, s = np.load(path)
    m = (f >= E_LO) & (f <= E_HI)
    ff, ss = f[m], s[m]
    noise = np.interp(0.8, f, s)
    pk, props = find_peaks(ss, height=10*noise, prominence=5*noise)
    E = ff[pk]
    print(f"\n{label}: {len(E)} features in [{E_LO},{E_HI}] meV")
    print("  " + ", ".join(f"{e:.3f}" for e in E))
    return E

def ratio_score(E):
    """fraction of pairwise ratios (within [RATIO_LO,RATIO_HI]) lying within
    TOL of tau or tau^2"""
    if len(E) < 2:
        return 0.0, 0, []
    r = np.array([b/a for i, a in enumerate(E) for b in E[i+1:]
                  if RATIO_LO <= b/a <= RATIO_HI])
    if len(r) == 0:
        return 0.0, 0, []
    hits = [(rv, name) for rv in r for name, t in targets.items()
            if abs(rv - t)/t <= TOL]
    return len(hits)/len(r), len(r), hits

def null_test(n_feat, obs_frac):
    """place n_feat features uniformly in the window, same scoring"""
    fracs = np.empty(N_NULL)
    for k in range(N_NULL):
        E = np.sort(rng.uniform(E_LO, E_HI, n_feat))
        fracs[k], _, _ = ratio_score(E)
    p = (fracs >= obs_frac).mean()
    return fracs.mean(), p

print("="*64)
print(f"TAU-RATIO TEST  (window {E_LO}-{E_HI} meV, tolerance +-{TOL*100:.0f}%)")
print("targets scored: tau=1.618, tau^2=2.618")
print("NOTE: 3/2, 5/3, 8/5 are reported but NOT scored - periodic box")
print("acoustics produce rational ratios, so Fibonacci neighbours are")
print("contaminated; tau itself is the clean target.")
print("="*64)

for path, label in [("md_wphase_vdos_total.npy",        "W-phase 3x2x1 (quasiperiodic motif)"),
                    ("md_xphase_10x4x3_vdos_total.npy", "X-phase 10x4x3 (negative control)"),
                    ("md_8x4x4_s21_vdos_total.npy",     "X-phase 8x4x4 (second control)")]:
    E = features(path, label)
    frac, n_r, hits = ratio_score(E)
    if n_r == 0:
        print("  too few features for ratios"); continue
    null_mean, p = null_test(len(E), frac)
    print(f"  pairwise ratios in range: {n_r}")
    print(f"  fraction within {TOL*100:.0f}% of tau or tau^2: {frac:.3f}")
    print(f"  null expectation (random features): {null_mean:.3f},  p = {p:.3f}")
    verdict = ("MORE tau-like than chance" if p < 0.05 else
               "consistent with chance")
    print(f"  -> {verdict}")
    if hits:
        print("  hits: " + ", ".join(f"{rv:.3f} ({nm})" for rv, nm in hits[:8]))
    # Fibonacci-neighbourhood ratios, reported for context only
    r = np.array([b/a for i, a in enumerate(E) for b in E[i+1:]
                  if RATIO_LO <= b/a <= RATIO_HI])
    for nm, t in fib.items():
        n_near = (np.abs(r - t)/t <= TOL).sum()
        if n_near:
            print(f"  (context: {n_near} ratios near {nm}={t:.3f})")

# ---------------- cross-approximant gap ratios ----------------
print("\n" + "="*64)
print("CROSS-APPROXIMANT: dispersion gap centres (from O3_dispersion)")
print("="*64)
g26_first, g60_first = 10.29, 8.36
g60 = [9.2, 12.0, 15.4]
print(f"first ZB gap 26-atom / 60-atom: {g26_first}/{g60_first} = {g26_first/g60_first:.3f}"
      f"  (cell-size ratio 1.27; tau = {TAU:.3f})")
print(f"60-atom gap-centre ratios: " +
      ", ".join(f"{b/a:.3f}" for a, b in zip(g60, g60[1:])) +
      f"  (Fibonacci drift toward tau would show these -> 1.618 as size grows)")
print("the 265-atom harmonic would be the third rung of this test (not run)")

TAU-RATIO TEST  (window 1.0-8.0 meV, tolerance +-3%)
targets scored: tau=1.618, tau^2=2.618
NOTE: 3/2, 5/3, 8/5 are reported but NOT scored - periodic box
acoustics produce rational ratios, so Fibonacci neighbours are
contaminated; tau itself is the clean target.

W-phase 3x2x1 (quasiperiodic motif): 78 features in [1.0,8.0] meV
  1.365, 1.530, 2.626, 2.657, 2.688, 2.740, 2.781, 2.843, 2.947, 2.998, 3.619, 3.681, 3.815, 4.012, 4.063, 4.115, 4.208, 4.332, 4.374, 4.405, 4.487, 4.539, 4.601, 4.663, 4.746, 4.828, 4.880, 4.932, 5.004, 5.035, 5.087, 5.139, 5.180, 5.221, 5.294, 5.397, 5.459, 5.542, 5.573, 5.656, 5.718, 5.811, 5.935, 6.069, 6.121, 6.193, 6.286, 6.338, 6.359, 6.410, 6.452, 6.534, 6.596, 6.658, 6.700, 6.741, 6.803, 6.886, 6.948, 7.000, 7.051, 7.103, 7.144, 7.206, 7.237, 7.289, 7.341, 7.362, 7.424, 7.496, 7.527, 7.568, 7.610, 7.682, 7.754, 7.827, 7.910, 7.972
  pairwise ratios in range: 2307
  fraction within 3% of tau or tau^2: 0.097
  null expectation (random features): 0.103, 